In [8]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [9]:
from typing import TypedDict,Annotated,List,Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel,Field
from langgraph.graph import END,StateGraph,START
from typing import List
import operator

#### State(Reflection)

In [10]:
grades=Literal[
    "ultra-conservative",
    "conservative",
    "moderate",
    "aggressive",
    "high risk"
]

class State(TypedDict):
    investment_plan:str
    investor_profile:str
    target_grade:grades
    feedback:str
    grade:grades
    n:int=0

- `investment_plan`: Our generated plan that will be evaluated and revised if need be
- `investor_profile`: The user inputted profile that we'll use as a reference for the plan
- `target_grade`: A generated ideal risk tolerance grade based on the investor profile
- `feedback`: Evaluator feedback for the investment plan
- `grade`: Evaluated grade of the investment plan
- `n`: Number of evaluation iterations

**Note:** Both grade fields are `Literal` meaning they can only take a value within the defined list.


#### Setup Node

In [11]:
grade_prompt=ChatPromptTemplate.from_messages([
    ("system",
     "You are an investor advisor. Given the investor's profile and their proposed plan,"
     "choose exactly one risk classifications from: ultra-conservative, conservative, moderate, aggressive, high-risk."
     "Return ONLY the grade"
    ),
    ("user",
     "Investor profile:\n\n{investor_profile}\n\n"
    )
])

grade_pipe=grade_prompt | llm

In [12]:
def determine_target_grade(state:State):
    """Ask the llm to pick the best-fitting target grade"""
    response=grade_pipe.invoke({
        "investor_profile":state["investor_profile"]
    })
    # return as a plain dict so langGraph can merge it into the state
    return {"target_grade":response.content.lower()}

In [13]:
# initiate empty state for the user inputted investor profile

dummy_state:State={
    "investor_plan":"",
    "investor_profile":(
        "Age: 29\n"
        "Salary: $110,000\n"
        "Assets: $40,000\n"
        "Goal: Achieve financial independence by age 45\n"
        "Risk tolerence: High"
    ),
    "target_grade": "",
    "feedback": "",
    "grade": "",
    "n": 0
}

In [14]:
from pprint import pprint
# get target grade 
target_grade= determine_target_grade(dummy_state)
# update target grade with the returned dict
dummy_state.update(target_grade)
pprint(dummy_state)

{'feedback': '',
 'grade': '',
 'investor_plan': '',
 'investor_profile': 'Age: 29\n'
                     'Salary: $110,000\n'
                     'Assets: $40,000\n'
                     'Goal: Achieve financial independence by age 45\n'
                     'Risk tolerence: High',
 'n': 0,
 'target_grade': 'aggressive'}
